# Matrix-Free Rayleigh Relaxation — Colab notebook

This notebook is a cleaned, paper-oriented Colab workflow for the revised matrix-free Schrödinger solver.

It is designed to:

- verify the installation and small correctness test,
- run the main validation examples,
- save results either locally or to Google Drive,
- and produce professional-looking plots for the paper.

Before running, choose **Runtime → Change runtime type → GPU**.


In [ ]:
# Check the GPU assigned by Colab.
!nvidia-smi


## 1. Get the repository

Use **Option A** after the repository is public on GitHub.
Until then, use **Option B** to upload the ZIP bundle.


In [ ]:
# OPTION A: clone the future public repository.
REPO_URL = "https://github.com/mahdi-sasar/matrix-free-rayleigh-relaxation.git"

# !rm -rf /content/matrix-free-rayleigh-relaxation /content/matrix_free_rayleigh_relaxation
# !git clone $REPO_URL /content/matrix-free-rayleigh-relaxation
# %cd /content/matrix-free-rayleigh-relaxation


In [ ]:
# OPTION B: upload the ZIP bundle if the repository is not yet public.
# Uncomment and run this cell.

# from google.colab import files
# import zipfile, shutil, pathlib
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# shutil.rmtree('/content/matrix_free_rayleigh_relaxation', ignore_errors=True)
# with zipfile.ZipFile(zip_name) as zf:
#     zf.extractall('/content')
# %cd /content/matrix_free_rayleigh_relaxation


## 2. Install lightweight dependencies

Do not reinstall TensorFlow in Colab unless absolutely necessary. The `--no-deps` flag is intentional.


In [ ]:
!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e . --no-deps


## 3. Optional: mount Google Drive

For serious or overnight runs, mount Google Drive so the output persists after runtime resets.


In [ ]:
# Uncomment if you want to save results to Drive.
# from google.colab import drive
# drive.mount('/content/drive')


## 4. Configure TensorFlow and plotting defaults


In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import Image, display

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Could not set memory growth:", exc)

tf.keras.backend.set_floatx("float64")
print("Default Keras float:", tf.keras.backend.floatx())

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": False,
})


## 5. Helper functions for robust plotting and output discovery

These functions let the notebook work whether the results are saved locally or in Google Drive.


In [ ]:
SEARCH_ROOTS = [
    Path("results"),
    Path("/content/drive/MyDrive/mrsr_results"),
    Path("/content/drive/My Drive/mrsr_results"),
]

def latest_csv(named):
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.glob(f"**/{named}"))
    if not candidates:
        raise FileNotFoundError(f"Could not find {named} under {SEARCH_ROOTS}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


def read_metadata(outdir):
    meta_path = Path(outdir) / "metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            return json.load(f)
    return {}


def infer_box_from_metadata(outdir, fallback=16.0):
    meta = read_metadata(outdir)
    return float(meta.get("box", fallback))


def nice_energy_plot(df, out_png=None, title=r"$\mathrm{H}_2^+$ finite-box potential-energy curve"):
    df = df.sort_values("R_Bohr").copy()
    idx = int(df["total_energy_Ry"].idxmin())
    rbest = float(df.loc[idx, "R_Bohr"])
    ebest = float(df.loc[idx, "total_energy_Ry"])

    fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
    ax.plot(df["R_Bohr"], df["electronic_energy_Ry"], marker="o", markersize=5, linewidth=2.0, label="Electronic energy")
    ax.plot(df["R_Bohr"], df["total_energy_Ry"], marker="s", markersize=5, linewidth=2.0, label="Total energy")
    ax.scatter([rbest], [ebest], s=90, zorder=5, label=fr"Lowest sampled total ($R={rbest:.3f}$ Bohr)")
    ax.axvline(rbest, linestyle="--", linewidth=1.0, alpha=0.5)
    ax.set_xlabel("Internuclear separation $R$ (Bohr)")
    ax.set_ylabel("Energy (Ry)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25, linewidth=0.8)
    ax.legend(frameon=True)
    if out_png:
        fig.savefig(out_png, bbox_inches="tight")
        fig.savefig(Path(out_png).with_suffix(".pdf"), bbox_inches="tight")
        print("Saved:", out_png)
        print("Saved:", Path(out_png).with_suffix(".pdf"))
    plt.show()
    return idx, rbest, ebest


def nice_heatmap(
    csv_path,
    box,
    out_png=None,
    density=True,
    title="Mid-slice density",
    cmap="YlOrRd",
    add_contours=True,
    xlabel="X (Bohr)",
    ylabel="Y (Bohr)",
    nuclei=None,
):
    arr = pd.read_csv(csv_path, header=None).values
    data = arr**2 if density else arr

    fig, ax = plt.subplots(figsize=(6.4, 5.4), constrained_layout=True)
    extent = [-box/2, box/2, -box/2, box/2]
    im = ax.imshow(data.T, origin="lower", extent=extent, aspect="equal", cmap=cmap)
    if add_contours and np.nanmax(data) > np.nanmin(data):
        xs = np.linspace(extent[0], extent[1], data.shape[0])
        ys = np.linspace(extent[2], extent[3], data.shape[1])
        X, Y = np.meshgrid(xs, ys, indexing="ij")
        levels = np.linspace(np.nanmin(data), np.nanmax(data), 14)[1:-1]
        ax.contour(X, Y, data, levels=levels, colors="white", linewidths=0.45, alpha=0.75)

    if nuclei is not None:
        nx, ny = zip(*nuclei)
        ax.scatter(nx, ny, marker="o", s=70, facecolor="none", edgecolor="black", linewidth=1.4, label="protons")
        ax.scatter(nx, ny, marker="+", s=90, color="black", linewidth=1.1)
        ax.legend(loc="upper right", frameon=True)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$|\psi|^2$" if density else r"$\psi$")
    if out_png:
        fig.savefig(out_png, bbox_inches="tight")
        fig.savefig(Path(out_png).with_suffix(".pdf"), bbox_inches="tight")
        print("Saved:", out_png)
        print("Saved:", Path(out_png).with_suffix(".pdf"))
    plt.show()


def h2plus_plane_info(outdir, R, plane="bond"):
    """Return slice path, labels, and optional 2D nuclei positions."""
    outdir = Path(outdir)
    meta = read_metadata(outdir)
    axis = meta.get("axis", "z")
    center = meta.get("molecular_midpoint_bohr", [0.0, 0.0, 0.0])
    cx, cy, cz = map(float, center)
    R = float(R)

    case_dir = outdir / f"R_{R:.4f}"

    if plane == "bond":
        if axis == "z":
            filename, labels = "psi_bond_xz.csv", ("X", "Z")
            nuclei = [(cx, cz - 0.5*R), (cx, cz + 0.5*R)]
        elif axis == "x":
            filename, labels = "psi_bond_xy.csv", ("X", "Y")
            nuclei = [(cx - 0.5*R, cy), (cx + 0.5*R, cy)]
        else:  # axis == "y"
            filename, labels = "psi_bond_xy.csv", ("X", "Y")
            nuclei = [(cx, cy - 0.5*R), (cx, cy + 0.5*R)]
    elif plane == "transverse":
        if axis == "z":
            filename, labels = "psi_transverse_xy.csv", ("X", "Y")
        elif axis == "x":
            filename, labels = "psi_transverse_yz.csv", ("Y", "Z")
        else:  # axis == "y"
            filename, labels = "psi_transverse_xz.csv", ("X", "Z")
        nuclei = None
    else:
        raise ValueError("plane must be 'bond' or 'transverse'.")

    path = case_dir / filename
    if not path.exists() and plane == "transverse":
        path = case_dir / "psi_mid_z.csv"  # compatibility with older runs

    return path, labels, nuclei


## 6. Small correctness test


In [ ]:
!pytest -q tests/test_small_dense_comparison.py


## 7. Validation example: 3D harmonic oscillator


In [ ]:
!python examples/run_harmonic_oscillator_3d.py     --n 48     --box 12.0     --alpha 0.25     --sigma 0.9     --tol 1e-8     --max-iter 50000     --out results/colab_harmonic_48


In [ ]:
pd.read_csv("results/colab_harmonic_48/history.csv").tail()


## 8. Hydrogen in a box


In [ ]:
!python examples/run_hydrogen_box.py     --n 64     --box 10.0     --init gaussian     --sigma 0.5     --tol 1e-8     --max-iter 100000     --out results/colab_hydrogen_64


In [ ]:
with open("results/colab_hydrogen_64/metadata.json") as f:
    h_meta = json.load(f)
h_meta


In [ ]:
# Professional hydrogen mid-slice heatmap.
h_box = infer_box_from_metadata("results/colab_hydrogen_64", fallback=10.0)
nice_heatmap(
    "results/colab_hydrogen_64/psi_mid_z.csv",
    box=h_box,
    out_png="results/colab_hydrogen_64/psi_mid_z_density_pretty.png",
    density=True,
    title="Hydrogen in a box: mid-slice probability density",
    cmap="YlOrRd",
)


## 9. H$_2^+$ quick curve run

This is the smaller, faster molecular run. For final paper figures, use the larger production template further below.


In [ ]:
!python examples/run_h2plus_curve.py     --n 48     --box 16.0     --r-values 0.8 1.0 1.2 1.4 1.6 1.8 2.0 2.2 2.5 3.0 3.5 4.0     --sigma 0.5     --tol 1e-7     --max-iter 80000     --check-every 20     --out results/colab_h2plus_curve_48


In [ ]:
# Load the most recent H2+ result, whether it is local or in Drive.
h2_csv = latest_csv("h2plus_curve.csv")
H2_OUT = h2_csv.parent
print("Using H2+ output folder:")
print(H2_OUT)
print()
print("Using CSV:")
print(h2_csv)

h2 = pd.read_csv(h2_csv)
display(h2)


## 10. Professional H$_2^+$ energy curve


In [ ]:
curve_png = H2_OUT / "h2plus_curve_pretty.png"
idx, R_best, E_best = nice_energy_plot(h2, out_png=curve_png)
print(f"Lowest sampled total-energy point: R = {R_best:.6f} Bohr, E_total = {E_best:.12f} Ry")


## 11. Publication-ready H$_2^+$ heatmaps

The updated H$_2^+$ script now saves **both**:

- a **bond-plane slice** containing both protons and the molecular axis;
- a **transverse midpoint slice** perpendicular to the bond.

For the paper, the bond-plane density plot is the most important visual because it directly shows the bonding electron density between the nuclei. The transverse slice is useful as a symmetry and grid-quality diagnostic.


In [ ]:
case_dir = H2_OUT / f"R_{R_best:.4f}"
box_h2 = infer_box_from_metadata(H2_OUT, fallback=16.0)

# Bond-plane density: contains both nuclei and the molecular bond.
bond_csv, bond_labels, nuclei_2d = h2plus_plane_info(H2_OUT, R_best, plane="bond")
bond_png = case_dir / "h2plus_bond_plane_density_pretty.png"

print("Bond-plane slice:", bond_csv)
nice_heatmap(
    bond_csv,
    box=box_h2,
    out_png=bond_png,
    density=True,
    title=fr"$\mathrm{{H}}_2^+$ bond-plane density at $R={R_best:.3f}$ Bohr",
    cmap="YlOrRd",
    xlabel=f"{bond_labels[0]} (Bohr)",
    ylabel=f"{bond_labels[1]} (Bohr)",
    nuclei=nuclei_2d,
)

# Transverse midpoint density: perpendicular to the molecular axis.
trans_csv, trans_labels, _ = h2plus_plane_info(H2_OUT, R_best, plane="transverse")
trans_png = case_dir / "h2plus_transverse_density_pretty.png"

print("Transverse slice:", trans_csv)
nice_heatmap(
    trans_csv,
    box=box_h2,
    out_png=trans_png,
    density=True,
    title=fr"$\mathrm{{H}}_2^+$ transverse midpoint density at $R={R_best:.3f}$ Bohr",
    cmap="YlOrRd",
    xlabel=f"{trans_labels[0]} (Bohr)",
    ylabel=f"{trans_labels[1]} (Bohr)",
    nuclei=None,
)


## 12. Optional: command-line publication plots

These cells use the packaged scripts to regenerate the same figures directly from the saved CSV files. They are useful when preparing the final figure files for the manuscript.


In [ ]:
# Replot the H2+ curve from CSV using the packaged script.
!python scripts/plot_h2plus_curve.py "$h2_csv" --out "$H2_OUT/h2plus_curve_script.png"

# Replot the best H2+ bond-plane and transverse slices using the packaged script.
!python scripts/plot_h2plus_slices.py "$H2_OUT" --R best --plane bond --out "$case_dir/h2plus_bond_plane_script.png" --pdf
!python scripts/plot_h2plus_slices.py "$H2_OUT" --R best --plane transverse --out "$case_dir/h2plus_transverse_script.png" --pdf


## 13. Production template: larger H$_2^+$ run to Google Drive

This is the paper-oriented run template. It is suitable for a paid Colab GPU and should be left to run for a longer time.


In [ ]:
# Uncomment after mounting Google Drive.
# !python examples/run_h2plus_curve.py #     --n 96 #     --box 20.0 #     --r-values 0.8 1.0 1.2 1.4 1.6 1.8 2.0 2.2 2.4 2.6 3.0 3.5 4.0 5.0 #     --sigma 0.5 #     --tol 1e-8 #     --max-iter 200000 #     --check-every 20 #     --out /content/drive/MyDrive/mrsr_results/h2plus_curve_n96_box20


## 14. Optional scaling run template


In [ ]:
!python scripts/run_scaling_hydrogen.py     --n-values 64 80 96 112 128     --box 10.0     --tol 1e-8     --sigma 0.5     --max-iter 100000     --out results/colab_hydrogen_scaling.csv

pd.read_csv("results/colab_hydrogen_scaling.csv")


## 15. Copy local results to Drive


In [ ]:
# Uncomment if Drive is mounted and you want to preserve local results.
# !mkdir -p /content/drive/MyDrive/mrsr_results
# !cp -r results/* /content/drive/MyDrive/mrsr_results/
